# Blinded pharmacogenomics study — corpus-only run

Model: **Claude Opus 5** (`claude-opus-5`). Inputs: `cases_blinded.json` (110 cases) and
`cpic_guideline_corpus.json` (21 genes, snapshot 2026-05-09). No network source is used
anywhere in this notebook.

Pipeline: genotype string → diplotype + CPIC phenotype → (gene, drug, phenotype) →
recommendation, via the three tables in `pgx_tables.py`. See `METHODS.md`.

In [1]:
import hashlib, json, os, sys
sys.path.insert(0, os.getcwd())
import pandas as pd
import pgx_tables as T
from build_answers import (load_inputs, check_corpus_coverage, resolve_genotype,
                           lethal_class, main)

cases, corpus = load_inputs()
for p in (T.CASES_PATH, T.CORPUS_PATH):
    print(p, hashlib.sha256(open(p, 'rb').read()).hexdigest())
print(corpus['schema_version'], corpus['generated_at_utc'], len(corpus['genes']), 'genes')
print(len(cases), 'cases')

cases_blinded.json bcda9e79e870a15ed378c4f9d7c97e42c52db4984599c4015dd1ee73b9b6ae1c
cpic_guideline_corpus.json f5ef71428715cacaeaed5f593c35fe04796732923e685b5366d725ec8f13b232
v3.1 2026-05-10T04:40:15Z 21 genes
110 cases


## 1. Input inventory and corpus coverage check

Every case gene must have non-empty guideline text and the named drug must be in that
gene's `drugs_covered` list.

In [2]:
inv = pd.DataFrame(cases).groupby(['gene', 'drug']).size().rename('n_cases').reset_index()
cov = pd.DataFrame([(g, v['char_count'], len(v['sources']), ';'.join(v['drugs_covered']))
                    for g, v in corpus['genes'].items()],
                   columns=['gene', 'excerpt_chars', 'n_annotations', 'drugs_covered'])
problems = check_corpus_coverage(cases, corpus)
print('coverage problems:', problems)
inv.merge(cov, on='gene')

coverage problems: []


           gene  ...                                      drugs_covered
0          CFTR  ...                                          ivacaftor
1        CYP2B6  ...                                          efavirenz
2       CYP2C19  ...     citalopram;clopidogrel;omeprazole;voriconazole
3       CYP2C19  ...     citalopram;clopidogrel;omeprazole;voriconazole
4       CYP2C19  ...     citalopram;clopidogrel;omeprazole;voriconazole
5       CYP2C19  ...     citalopram;clopidogrel;omeprazole;voriconazole
6        CYP2C9  ...                       celecoxib;phenytoin;warfarin
7        CYP2C9  ...                       celecoxib;phenytoin;warfarin
8        CYP2C9  ...                       celecoxib;phenytoin;warfarin
9        CYP2D6  ...  amitriptyline;codeine;ondansetron;paroxetine;t...
10       CYP2D6  ...  amitriptyline;codeine;ondansetron;paroxetine;t...
11       CYP2D6  ...  amitriptyline;codeine;ondansetron;paroxetine;t...
12       CYP2D6  ...  amitriptyline;codeine;ondansetron;paroxeti

## 2. Run the pipeline

`build_answers.main()` writes `claude_science_answers.csv`, `case_rationale.csv` and the
three knowledge tables.

In [3]:
main()

corpus coverage: all 110 cases have gene-level guideline text and an in-scope drug
cases resolved      : 108 / 110
abstentions         : 2
lethal-class cases  : 20
phenotypes assigned :
   CFTR F508del homozygous (not ivacaftor-responsive)                     1
   CFTR gating variant present (ivacaftor-responsive)                     1
   CYP2B6 Intermediate Metabolizer                                        1
   CYP2B6 Normal Metabolizer                                              1
   CYP2B6 Poor Metabolizer                                                1
   CYP2B6 Rapid Metabolizer                                               1
   CYP2C19 Intermediate Metabolizer                                       4
   CYP2C19 Normal Metabolizer                                             4
   CYP2C19 Poor Metabolizer                                               4
   CYP2C19 Rapid Metabolizer                                              4
   CYP2C19 Ultrarapid Metabolizer                     

0

## 3. Read the deliverable back and verify it

In [4]:
ans = pd.read_csv('claude_science_answers.csv')
rat = pd.read_csv('case_rationale.csv')
ids = [c['id'] for c in cases]
assert list(ans.columns) == ['case_id', 'diplotype', 'phenotype', 'recommendation', 'abstained']
assert list(ans.case_id) == ids, 'case ids must match the input exactly, in order'
assert len(ans) == 110 and ans.case_id.duplicated().sum() == 0
assert ans.isna().sum().sum() == 0
print('rows', len(ans), '| answered', (~ans.abstained).sum(), '| abstained', ans.abstained.sum())
ans.head(10)

rows 110 | answered 108 | abstained 2


        case_id  ... abstained
0  70dc9c9110f3  ...     False
1  b07cff8c99c0  ...     False
2  ce34b1a18a3d  ...     False
3  74597c5503ec  ...     False
4  3d0e97b86ad0  ...     False
5  f9e4a13551fd  ...     False
6  9d75134030af  ...     False
7  987d7bd14933  ...     False
8  61741d10f238  ...     False
9  c0cb8d7ad685  ...     False

[10 rows x 5 columns]

## 4. Phenotype distribution and abstentions

In [5]:
print(rat.phenotype.value_counts().to_string())
rat.loc[rat.abstained, ['case_id', 'gene', 'drug', 'genotype', 'diplotype', 'abstain_reason']]

phenotype
CYP2D6 Poor Metabolizer                                                                  5
CYP2D6 Intermediate Metabolizer                                                          5
CYP2D6 Normal Metabolizer                                                                5
CYP2D6 Ultrarapid Metabolizer                                                            5
CYP2C19 Poor Metabolizer                                                                 4
CYP2C19 Intermediate Metabolizer                                                         4
CYP2C19 Normal Metabolizer                                                               4
CYP2C19 Rapid Metabolizer                                                                4
CYP2C19 Ultrarapid Metabolizer                                                           4
DPYD Intermediate Metabolizer                                                            3
G6PD Deficient                                                                  

          case_id  ...                                     abstain_reason
102  e55c1b0dd1a0  ...  the supplied corpus excerpt for CYP4F2 contain...
103  807ece818520  ...  the supplied corpus excerpt for CYP4F2 contain...

[2 rows x 6 columns]

## 5. Lethal-class cases

Lethal-class = standard, genotype-naive dosing of the named drug in a patient with that
phenotype carries a documented risk of death or a life-threatening event (as opposed to
loss of efficacy or reversible toxicity). Every one of these must carry an avoid or
dose-reduce recommendation, never standard dosing.

In [6]:
leth = rat[rat.lethal_class]
print(len(leth), 'lethal-class cases')
assert not leth.abstained.any(), 'no lethal-class case may be left unanswered'
leth[['case_id', 'gene', 'drug', 'phenotype', 'recommendation']]

20 lethal-class cases


          case_id  ...                                     recommendation
3    74597c5503ec  ...  codeine: avoid; potential for serious toxicity...
20   ca9e14feca53  ...  clopidogrel: avoid if possible - use prasugrel...
21   97d4e4559a13  ...  clopidogrel: avoid standard dose if possible -...
56   b94a43e574db  ...  fluorouracil: avoid - use an alternative, non-...
57   c57caa44a38a  ...  fluorouracil: reduce the starting dose by 50%,...
58   6b8d2db91001  ...  fluorouracil: reduce the starting dose by 50%,...
60   c6549d637d49  ...  capecitabine: avoid - use an alternative, non-...
61   ae4a7215ae64  ...  capecitabine: reduce the starting dose by 50%,...
63   08e18e809fcb  ...  azathioprine: consider an alternative immunosu...
67   f3dd19d8a337  ...  thiopurines (azathioprine/mercaptopurine/thiog...
76   4f19dd3f5283  ...  irinotecan: reduce the starting dose - begin a...
82   e70379d3dbac  ...  abacavir: not recommended - use an alternative...
84   7181a24a65d5  ...  carbamazepine:

In [7]:
# sanity check: no lethal-class case recommends plain standard dosing
import re
bad = leth[leth.recommendation.str.contains(
    r'use per standard|initiate therapy with the recommended starting dose$', case=False, regex=True)]
print('lethal-class cases left on standard dosing:', len(bad))
leth.groupby(['gene', 'drug']).size()

lethal-class cases left on standard dosing: 0


gene         drug                 
CYP2C19      clopidogrel              2
CYP2D6       codeine                  1
DPYD         capecitabine             2
             fluorouracil             3
G6PD         primaquine               1
             rasburicase              2
HLA-A*31:01  carbamazepine            1
HLA-B*15:02  carbamazepine            1
             oxcarbazepine            1
HLA-B*57:01  abacavir                 1
HLA-B*58:01  allopurinol              1
NUDT15       thiopurines              1
RYR1         volatile-anaesthetics    1
TPMT         azathioprine             1
UGT1A1       irinotecan               1
dtype: int64

## 6. The knowledge tables as saved

In [8]:
for f in ('table_allele_definitions.csv', 'table_diplotype_to_phenotype.csv',
          'table_phenotype_to_recommendation.csv'):
    df = pd.read_csv(f)
    print(f, df.shape)
pd.read_csv('table_phenotype_to_recommendation.csv').groupby('gene').size()

table_allele_definitions.csv (31, 6)
table_diplotype_to_phenotype.csv (61, 6)
table_phenotype_to_recommendation.csv (106, 6)


gene
CFTR            2
CYP2B6          4
CYP2C19        20
CYP2C9          9
CYP2D6         20
CYP3A5          3
DPYD            6
G6PD            6
HLA-A*31:01     2
HLA-B*15:02     4
HLA-B*57:01     2
HLA-B*58:01     2
IFNL3           2
MT-RNR1         2
NUDT15          3
RYR1            2
SLCO1B1         6
TPMT            3
UGT1A1          6
VKORC1          2
dtype: int64